# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jawad-ahmed-developer/flyRank_Internship_Tasks/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

ML-04 already selected the five features for the Refresh / Content Opportunity Scoring lane. I am not selecting new features here.

The approved feature vector is:

1. `imp_prev30`
2. `clicks_prev30`
3. `avg_position_prev30`
4. `content_age_days`
5. `days_since_last_update`

The first three are constructed from the 30 days before the March 31 decision point.

The final two describe the content state at the decision point.

Identifiers are retained separately for grouping and auditing but are not included in `X`.

No future outcome information is used to construct the final feature vector.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

%pip -q install duckdb huggingface_hub

In [14]:
import os
import getpass
import duckdb
import pandas as pd
import numpy as np

# Token order:
# environment variable -> Colab Secret -> prompt as last resort
HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

if not HF_TOKEN:
    HF_TOKEN = getpass.getpass(
        "Paste your Hugging Face READ token (hf_...): "
    )

Paste your Hugging Face READ token (hf_...): ··········


In [15]:
con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

# Exact file/directory structure from the working warehouse notebook.
TABLES = {
    "dim_clients":
        f"read_parquet('{REL}/dim_clients.parquet')",

    "dim_content":
        f"read_parquet('{REL}/dim_content.parquet')",

    "fact_daily":
        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",

    "fact_daily_sample":
        f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",

    "fact_query_90d":
        f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

print("Connected to FlyRank warehouse.")
print("Development decision point: 2026-03-31")

Connected to FlyRank warehouse.
Development decision point: 2026-03-31


In [16]:
daily_schema = con.sql(
    f"DESCRIBE SELECT * FROM {TABLES['fact_daily']}"
).df()

content_schema = con.sql(
    f"DESCRIBE SELECT * FROM {TABLES['dim_content']}"
).df()

daily_columns = daily_schema["column_name"].tolist()
content_columns = content_schema["column_name"].tolist()

print("Daily fact columns:")
print(daily_columns)

print("\nContent dimension columns:")
print(content_columns)

Daily fact columns:
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']

Content dimension columns:
['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'word_count', 'last_optimized_date', 'optim

In [17]:
required_daily_columns = [
    "client_hash_id",
    "content_hash_id",
    "report_date",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
]

missing_daily_columns = [
    col for col in required_daily_columns
    if col not in daily_columns
]

if missing_daily_columns:
    raise ValueError(
        f"Required daily columns are missing: {missing_daily_columns}"
    )

print("All required daily feature columns are present.")

All required daily feature columns are present.


In [18]:
CONTENT_CREATED = "content_created_date"
CONTENT_UPDATED = "content_updated_date"

print("Content creation column:", CONTENT_CREATED)
print("Content update column:", CONTENT_UPDATED)

Content creation column: content_created_date
Content update column: content_updated_date


In [19]:
DECISION_DATE = "2026-03-31"

date_check = con.sql(
    f"""
    SELECT
        COUNT(*) AS total_rows,

        SUM(
            CASE
                WHEN CAST(content_created_date AS DATE)
                     > DATE '{DECISION_DATE}'
                THEN 1
                ELSE 0
            END
        ) AS created_after_decision,

        SUM(
            CASE
                WHEN CAST(content_updated_date AS DATE)
                     > DATE '{DECISION_DATE}'
                THEN 1
                ELSE 0
            END
        ) AS updated_after_decision,

        MIN(CAST(content_created_date AS DATE)) AS earliest_created,
        MAX(CAST(content_created_date AS DATE)) AS latest_created,

        MIN(CAST(content_updated_date AS DATE)) AS earliest_updated,
        MAX(CAST(content_updated_date AS DATE)) AS latest_updated

    FROM {TABLES['dim_content']}
    """
).df()

date_check

,total_rows,created_after_decision,updated_after_decision,earliest_created,latest_created,earliest_updated,latest_updated
0,519606,86172.0,382739.0,2024-10-16,2026-07-06,2024-10-28,2026-07-06


In [20]:
content_state = con.sql(
    f"""
    SELECT
        content_hash_id,

        DATE_DIFF(
            'day',
            CAST(content_created_date AS DATE),
            DATE '{DECISION_DATE}'
        ) AS content_age_days

    FROM {TABLES['dim_content']}

    WHERE CAST(content_created_date AS DATE)
          <= DATE '{DECISION_DATE}'
    """
).df()

print(f"Content rows existing by decision date: {len(content_state):,}")

content_state.head()

Content rows existing by decision date: 433,434


,content_hash_id,content_age_days
0,content_004e9c4c32e88631,175
1,content_0236ef736698e17c,175
2,content_025f6cfd3c298870,175
3,content_0263d5f9b7a2ecd4,175
4,content_02752c6c1c60161f,175


In [21]:
content_update_state = con.sql(
    f"""
    SELECT
        content_hash_id,

        CASE
            WHEN CAST(content_updated_date AS DATE)
                 <= DATE '{DECISION_DATE}'
            THEN DATE_DIFF(
                'day',
                CAST(content_updated_date AS DATE),
                DATE '{DECISION_DATE}'
            )
            ELSE NULL
        END AS days_since_last_update

    FROM {TABLES['dim_content']}

    WHERE CAST(content_created_date AS DATE)
          <= DATE '{DECISION_DATE}'
    """
).df()

content_update_state.head()

,content_hash_id,days_since_last_update
0,content_004e9c4c32e88631,<NA>
1,content_0236ef736698e17c,<NA>
2,content_025f6cfd3c298870,<NA>
3,content_0263d5f9b7a2ecd4,<NA>
4,content_02752c6c1c60161f,<NA>


In [22]:
content_state = content_state.merge(
    content_update_state,
    on="content_hash_id",
    how="left"
)

print("Content-state shape:", content_state.shape)

content_state.head()

Content-state shape: (433434, 3)


,content_hash_id,content_age_days,days_since_last_update
0,content_004e9c4c32e88631,175,<NA>
1,content_0236ef736698e17c,175,<NA>
2,content_025f6cfd3c298870,175,<NA>
3,content_0263d5f9b7a2ecd4,175,<NA>
4,content_02752c6c1c60161f,175,<NA>


In [23]:
print("Missing values:")
print(
    content_state[
        ["content_age_days", "days_since_last_update"]
    ].isna().sum()
)

print("\nNegative values:")
print(
    (
        content_state[
            ["content_age_days", "days_since_last_update"]
        ] < 0
    ).sum()
)

Missing values:
content_age_days               0
days_since_last_update    296567
dtype: int64

Negative values:
content_age_days          0
days_since_last_update    0
dtype: Int64


In [24]:
missing_update = (
    content_state["days_since_last_update"].isna().sum()
)

total_content = len(content_state)

print(
    f"Missing days_since_last_update: "
    f"{missing_update:,} / {total_content:,}"
)

print(
    "Missing percentage:",
    round(missing_update / total_content * 100, 2),
    "%"
)

Missing days_since_last_update: 296,567 / 433,434
Missing percentage: 68.42 %


In [25]:
FEATURES = [
    "imp_prev30",
    "clicks_prev30",
    "avg_position_prev30",
    "content_age_days",
    "days_since_last_update",
]

feature_frame = historical_features.merge(
    content_state,
    on="content_hash_id",
    how="left"
)

feature_frame = feature_frame[
    [
        "client_hash_id",
        "content_hash_id",
        *FEATURES
    ]
].copy()

print("Feature frame shape:", feature_frame.shape)

print("\nFeature columns:")
print(FEATURES)

feature_frame.head()

Feature frame shape: (331436, 7)

Feature columns:
['imp_prev30', 'clicks_prev30', 'avg_position_prev30', 'content_age_days', 'days_since_last_update']


,client_hash_id,content_hash_id,imp_prev30,clicks_prev30,avg_position_prev30,content_age_days,days_since_last_update
0,client_62f4a7e64f5e0096,content_5d562ebeda84bdc7,773.0,0.0,11.467012,249.0,<NA>
1,client_62f4a7e64f5e0096,content_d6f555d072e070be,25534.0,72.0,4.089332,249.0,<NA>
2,client_62f4a7e64f5e0096,content_dabbfdf80d8773f0,1448.0,9.0,4.669199,249.0,<NA>
3,client_62f4a7e64f5e0096,content_a761d62e362213d3,5595.0,24.0,4.188382,249.0,<NA>
4,client_62f4a7e64f5e0096,content_eb6738715ef9bac1,412.0,2.0,9.730000,249.0,<NA>


In [26]:
duplicate_decision_rows = (
    feature_frame
    .groupby(["client_hash_id", "content_hash_id"])
    .size()
    .reset_index(name="row_count")
    .query("row_count > 1")
)

print(
    "Duplicate client/content feature rows:",
    len(duplicate_decision_rows)
)

assert len(duplicate_decision_rows) == 0

print("Feature-frame grain check passed.")

Duplicate client/content feature rows: 0
Feature-frame grain check passed.


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

The approved feature vector contains five numeric features. No categorical feature is included, so no categorical encoding is required in this notebook.

### 1. `imp_prev30` — previous-30-day impressions

This is the number of Google Search Console impressions observed for the content item during the previous 30-day window before the March 31, 2026 decision point.

- **Meaning:** historical search visibility.
- **Missing handling:** no additional fill is applied here; the feature was constructed from the available historical performance rows.
- **Available when?** Yes. It belongs to the historical window before the decision point, so it is available before the future outcome is evaluated.

### 2. `clicks_prev30` — previous-30-day clicks

This is the number of Google Search Console clicks observed during the previous 30-day window.

- **Meaning:** historical search traffic generated from Google Search.
- **Missing handling:** no additional fill is applied here; the feature comes from the historical performance aggregation.
- **Available when?** Yes. It comes from the previous 30-day window and is therefore known before the prediction moment.

### 3. `avg_position_prev30` — previous-30-day average position

This is the average Google Search Console search position calculated over the previous 30-day window.

- **Meaning:** historical search ranking performance.
- **Missing handling:** no additional fill is applied here; the value is retained from the historical aggregation.
- **Available when?** Yes. It describes performance before the decision point and does not use the future outcome window.

### 4. `content_age_days` — content age at the decision point

This is the number of days between `content_created_date` and March 31, 2026.

- **Meaning:** how old the content was when the decision would have been made.
- **Missing handling:** none in the constructed content-state table. The March 31 eligibility filter produced 0 missing values.
- **Available when?** Yes, because the creation date is already known for content that existed by the decision point.

### 5. `days_since_last_update` — time since the recorded update date

This is the number of days between `content_updated_date` and March 31, 2026, when the recorded update date is on or before the decision point.

- **Meaning:** how recently the content had been updated at the decision point.
- **Missing handling:** 296,567 of 433,434 eligible content rows (68.42%) are missing. These values are not replaced with zero because zero would incorrectly mean that the content was updated on the decision date. The missingness reflects that the current warehouse snapshot does not provide a valid pre-March-31 update state for those rows.
- **Available when?** Only when the recorded update date is on or before the decision point. Future update dates are not used as historical information.

### Categorical handling

No categorical feature is part of the approved five-feature vector, so no categorical encoding is performed.

### Overall availability rule

The feature vector is intended to contain only information that could be known at the March 31, 2026 decision point. Future outcome information is not used as a feature. IDs remain context for joining and identifying rows and are not included in the model feature vector.

In [27]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# summarize availability and missingness of the five approved features.

feature_availability = pd.DataFrame({
    "feature": FEATURES,
    "missing_rows": [
        feature_frame[f].isna().sum()
        for f in FEATURES
    ]
})

feature_availability["total_rows"] = len(feature_frame)

feature_availability["missing_pct"] = (
    feature_availability["missing_rows"]
    / feature_availability["total_rows"]
    * 100
)

feature_availability

,feature,missing_rows,total_rows,missing_pct
0,imp_prev30,0,331436,0.000000
1,clicks_prev30,0,331436,0.000000
2,avg_position_prev30,156625,331436,47.256484
3,content_age_days,2124,331436,0.640848
4,days_since_last_update,293357,331436,88.510904


In [28]:
print("Categorical features:")
print("None — all five approved features are numeric.")

print("\nNegative values:")
print(
    (
        feature_frame[FEATURES] < 0
    ).sum()
)

Categorical features:
None — all five approved features are numeric.

Negative values:
imp_prev30                0
clicks_prev30             0
avg_position_prev30       0
content_age_days          0
days_since_last_update    0
dtype: Int64


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.